# Task 2 — Features-Only Models (no data leakage)

Six tuned models using only player behaviour features.

**How leakage is prevented**
| Risk | Safeguard |
|---|---|
| `player_id` (it leaked the label in the original dataset; fixed by the organisers on 13 Nov) | Never a feature; `cv_fit_in_fold` raises an error if it appears |
| Feature statistics seeing validation rows | `FeatureBuilder` is fitted on each fold's training part only |
| Tuning overfitting the reported score | Tuned on separate folds (`tune_models.py`, seed 2026); scored here on seed 42 |
| Blend and class weights fitted on scored rows | Nested CV: fitted on 4 folds, scored on the 5th |
| Using the test set | Test rows are only transformed and predicted, never fitted on |

1. Tuned parameters
2. Compare six models on the same 5 folds
3. Blending and class weights (nested CV)
4. Choose the final configuration
5. Fit on all training data → write the submission

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
from sklearn.metrics import f1_score, log_loss, classification_report, confusion_matrix

from data import load_train, load_test, TARGET, CLASS_NAMES, TASK_DIR
from evaluate import cv_fit_in_fold, get_folds
from features import FeatureBuilder
from model_zoo import MODEL_VIEW, VIEWS, build
from postprocess import blend, fit_blend_weights, fit_class_weights, apply_class_weights, nested_score
from submission import save_submission

ART = TASK_DIR / "artifacts"; (ART / "oof").mkdir(parents=True, exist_ok=True)
ORDER = list(CLASS_NAMES.values())
train, test = load_train(), load_test()
y = train[TARGET].to_numpy()
folds = get_folds(y)

## 1. Tuned parameters (from `src/tune_models.py`)

In [2]:
tuned = json.loads((ART / "tuned_params.json").read_text())
display(pd.DataFrame({n: {"tuning F1 (3-fold)": round(r["tune_f1"], 4), "trials": r["n_trials"], "params": r["params"]}
                      for n, r in tuned.items()}).T)

,tuning F1 (3-fold),trials,params
AvgHistNB,0.7562,30,"{'bins': 'wide', 'alpha': 4.430317433764992}"
SplineLR,0.7541,5,"{'C': 4.394842165614594, 'n_knots': 4}"
LightGBM,0.7546,24,"{'learning_rate': 0.05, 'n_estimators': 353, '..."
XGBoost,0.754,9,"{'learning_rate': 0.05, 'n_estimators': 490, '..."
CatBoost,0.7536,18,"{'learning_rate': 0.05, 'iterations': 1991, 'd..."
RandomForest,0.7443,11,"{'max_depth': 17, 'min_samples_leaf': 3, 'max_..."


## 2. Compare models on the same 5 folds

In [3]:
oof, scores = {}, {}
for name in MODEL_VIEW:
    print(f"{name:13s}", end=" ")
    oof[name], scores[name], fold_f1 = cv_fit_in_fold(build(name, tuned[name]["params"]), train, y, VIEWS[MODEL_VIEW[name]], folds)
    np.save(ART / "oof" / f"noleak_{name}.npy", oof[name])
table = pd.DataFrame({n: {"macro F1": scores[n], "log-loss": log_loss(y, oof[n])} for n in oof}).T.sort_values("macro F1", ascending=False)
display(table.round(4))

AvgHistNB     

macro F1 0.7562  (folds 0.7562 ± 0.0018)  [33 feats, 19s]
SplineLR      

macro F1 0.7548  (folds 0.7548 ± 0.0021)  [33 feats, 43s]
LightGBM      

macro F1 0.7537  (folds 0.7537 ± 0.0020)  [41 feats, 17s]
XGBoost       

macro F1 0.7544  (folds 0.7544 ± 0.0020)  [41 feats, 46s]
CatBoost      

macro F1 0.7536  (folds 0.7536 ± 0.0025)  [41 feats, 29s]
RandomForest  

macro F1 0.7452  (folds 0.7452 ± 0.0036)  [41 feats, 161s]


,macro F1,log-loss
AvgHistNB,0.7562,0.6246
SplineLR,0.7548,0.6212
XGBoost,0.7544,0.6077
LightGBM,0.7537,0.6086
CatBoost,0.7536,0.6148
RandomForest,0.7452,0.6461


## 3. Blending and class weights (nested CV)
Fold-to-fold variation is about ±0.003, so differences below about 0.002 are noise.

In [4]:
ranked = list(table.index)
candidates = {f"{ranked[0]} alone": ranked[:1], "Top 2": ranked[:2], "Top 3": ranked[:3], "All 6": ranked}
candidates["Naive Bayes + best tree model"] = ["AvgHistNB", next(m for m in ranked if m in ("LightGBM", "XGBoost", "CatBoost"))]
rows = []
for cname, members in candidates.items():
    ps = [oof[m] for m in members]
    rows.append({"candidate": cname, "members": ", ".join(members),
                 "blend only": nested_score(ps, y, folds, use_cw=False),
                 "blend + class weights": nested_score(ps, y, folds, use_cw=True)})
nested = pd.DataFrame(rows).set_index("candidate")
display(nested.round(4))

,members,blend only,blend + class weights
candidate,,,
AvgHistNB alone,AvgHistNB,0.7562,0.7564
Top 2,"AvgHistNB, SplineLR",0.7563,0.7566
Top 3,"AvgHistNB, SplineLR, XGBoost",0.7553,0.7555
All 6,"AvgHistNB, SplineLR, XGBoost, LightGBM, CatBoo...",0.7555,0.7559
Naive Bayes + best tree model,"AvgHistNB, XGBoost",0.7551,0.7558


## 4. Choose the final configuration

In [5]:
# Best nested score; on near-ties (within 0.001) prefer fewer models and no class weights.
flat = nested[["blend only", "blend + class weights"]].stack().rename("score").reset_index()
flat.columns = ["candidate", "post", "score"]
flat["complexity"] = flat["candidate"].map(lambda c: len(candidates[c])) + (flat["post"] == "blend + class weights")
choice = flat[flat["score"] >= flat["score"].max() - 0.001].sort_values(["complexity", "score"], ascending=[True, False]).iloc[0]
members, use_cw = candidates[choice["candidate"]], choice["post"] == "blend + class weights"
print(f"FINAL: {choice['candidate']} ({choice['post']})   nested macro F1 = {choice['score']:.4f}   members: {members}")

ps = [oof[m] for m in members]
w = fit_blend_weights(ps, y) if len(ps) > 1 else np.array([1.0])
cw = fit_class_weights(blend(ps, w), y)[0] if use_cw else np.ones(4)
print("blend weights:", dict(zip(members, w.round(3).tolist())), " class weights:", cw.round(3).tolist())
oof_pred = apply_class_weights(blend(ps, w), cw)
print(classification_report(y, oof_pred, target_names=ORDER, digits=4))
display(pd.DataFrame(confusion_matrix(y, oof_pred, normalize="true"), index=ORDER, columns=ORDER).round(3))

FINAL: AvgHistNB alone (blend only)   nested macro F1 = 0.7562   members: ['AvgHistNB']
blend weights: {'AvgHistNB': 1.0}  class weights: [1.0, 1.0, 1.0, 1.0]
              precision    recall  f1-score   support

      Casual     0.7942    0.8534    0.8227     40957
     Grinder     0.7524    0.7358    0.7440     26032
      Social     0.7425    0.6841    0.7121     21029
       Whale     0.7663    0.7270    0.7461     15982

    accuracy                         0.7703    104000
   macro avg     0.7639    0.7501    0.7562    104000
weighted avg     0.7690    0.7703    0.7689    104000



,Casual,Grinder,Social,Whale
Casual,0.853,0.088,0.052,0.007
Grinder,0.185,0.736,0.030,0.050
Social,0.174,0.048,0.684,0.094
Whale,0.036,0.107,0.130,0.727


## 5. Fit on all training data → submission

In [6]:
test_probs = []
for m in members:
    fb = FeatureBuilder(**VIEWS[MODEL_VIEW[m]]).fit(train)
    Xtr, Xte = fb.transform(train), fb.transform(test)
    assert "player_id" not in Xtr.columns and list(Xtr.columns) == list(Xte.columns)
    test_probs.append(build(m, tuned[m]["params"]).fit(Xtr, y).predict_proba(Xte))
test_pred = apply_class_weights(blend(test_probs, w), cw)

display(pd.DataFrame({"train (true)": pd.Series(y).map(CLASS_NAMES).value_counts(normalize=True),
                      "OOF predicted": pd.Series(oof_pred).map(CLASS_NAMES).value_counts(normalize=True),
                      "test predicted": pd.Series(test_pred).map(CLASS_NAMES).value_counts(normalize=True)}).loc[ORDER].round(3))
full = save_submission(test_pred, "noleak")
print("saved submissions/task2_noleak.csv and submissions/full_noleak.csv")
full.head()

,train (true),OOF predicted,test predicted
Casual,0.394,0.423,0.425
Grinder,0.250,0.245,0.246
Social,0.202,0.186,0.184
Whale,0.154,0.146,0.145


saved submissions/task2_noleak.csv and submissions/full_noleak.csv


,id,task1,task2,task3,task4,task5
0,ANS00001,1,3,0.0,0,1
1,ANS00002,1,3,0.0,0,1
2,ANS00003,1,2,0.0,0,1
3,ANS00004,1,1,0.0,0,1
4,ANS00005,1,0,0.0,0,1
